In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder


In [2]:
df = pd.read_csv("synthetic_task_dataset.csv")
print(df.shape)
df.head()
df.columns

(1500, 10)


Index(['task_id', 'task_title', 'task_description', 'priority', 'deadline',
       'assigned_to', 'status', 'category', 'estimated_hours', 'created_at'],
      dtype='object')

In [3]:

df = df.dropna(subset=['task_description', 'category'])


le = LabelEncoder()
df['category_encoded'] = le.fit_transform(df['category'])


In [4]:
X = df['task_description']
y = df['category_encoded']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [5]:
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)



In [6]:
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
y_pred_nb = nb_model.predict(X_test_tfidf)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))


Naive Bayes Accuracy: 0.18666666666666668
              precision    recall  f1-score   support

           0       0.16      0.16      0.16        62
           1       0.23      0.11      0.15        56
           2       0.17      0.28      0.22        64
           3       0.21      0.25      0.22        61
           4       0.20      0.12      0.15        57

    accuracy                           0.19       300
   macro avg       0.19      0.18      0.18       300
weighted avg       0.19      0.19      0.18       300



In [7]:
svm_model = LinearSVC()
svm_model.fit(X_train_tfidf, y_train)
y_pred_svm = svm_model.predict(X_test_tfidf)

print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))


SVM Accuracy: 0.18
              precision    recall  f1-score   support

           0       0.16      0.15      0.15        62
           1       0.17      0.20      0.18        56
           2       0.17      0.17      0.17        64
           3       0.23      0.25      0.24        61
           4       0.16      0.14      0.15        57

    accuracy                           0.18       300
   macro avg       0.18      0.18      0.18       300
weighted avg       0.18      0.18      0.18       300



In [8]:
import numpy as np
y_pred_final = np.where(y_pred_svm == y_pred_nb, y_pred_svm, y_pred_svm)  # favor SVM
print("Final Accuracy:", accuracy_score(y_test, y_pred_final))


Final Accuracy: 0.18


In [9]:
import joblib
joblib.dump(vectorizer, 'tfidf_vectorizer.joblib')
joblib.dump(nb_model, 'naive_bayes_model.joblib')
joblib.dump(svm_model, 'svm_model.joblib')
joblib.dump(le, 'label_encoder.joblib')


['label_encoder.joblib']

In [10]:
desc = ["Fix urgent bug in API endpoint"]
desc_tfidf = vectorizer.transform(desc)
pred = svm_model.predict(desc_tfidf)
print("Predicted Category:", le.inverse_transform(pred))


Predicted Category: ['Bug Fix']


In [11]:
df['full_text'] = df['task_title'] + ' ' + df['task_description']


In [12]:
svm_model = LinearSVC(class_weight='balanced')


In [13]:
print(df['category'].value_counts())
print(df['category'].nunique())


category
Feature          319
Bug Fix          309
Testing          305
UI/UX            286
Documentation    281
Name: count, dtype: int64
5


In [14]:
df['full_text'] = df['task_title'].fillna('') + ' ' + df['task_description'].fillna('')
X = df['full_text']


In [15]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(df['category'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [16]:
print("Unique categories:", df['category'].nunique())
print("Top 5 categories:")
print(df['category'].value_counts().head())

print("\nExample descriptions:")
print(df['task_description'].head())


Unique categories: 5
Top 5 categories:
category
Feature          319
Bug Fix          309
Testing          305
UI/UX            286
Documentation    281
Name: count, dtype: int64

Example descriptions:
0    Property agreement position. Decide hundred cl...
1    Listen forward young common nice image. Yeah v...
2    Require friend here back project accept party ...
3    Director service kid want.\nThird fish hospita...
4    Question difficult professor plan southern. My...
Name: task_description, dtype: object


In [17]:
print(df['category'].value_counts().head())
print(df['task_description'].head())
print(df['task_title'].head())


category
Feature          319
Bug Fix          309
Testing          305
UI/UX            286
Documentation    281
Name: count, dtype: int64
0    Property agreement position. Decide hundred cl...
1    Listen forward young common nice image. Yeah v...
2    Require friend here back project accept party ...
3    Director service kid want.\nThird fish hospita...
4    Question difficult professor plan southern. My...
Name: task_description, dtype: object
0              Opportunity item indicate.
1                      Wish article hand.
2                          Dinner system.
3                      Sort right parent.
4    Type knowledge question there drive.
Name: task_title, dtype: object


In [18]:
df['text'] = df['task_title'] + ' ' + df['task_description']
X = df['text']
y = df['category']


In [19]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000,
    ngram_range=(1, 2)
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


In [20]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)
y_pred_nb = nb.predict(X_test_tfidf)
print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))

# SVM
svm = LinearSVC(class_weight='balanced')
svm.fit(X_train_tfidf, y_train)
y_pred_svm = svm.predict(X_test_tfidf)
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))

# Optional: Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_test_tfidf)
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))


Naive Bayes Accuracy: 0.19333333333333333
SVM Accuracy: 0.21333333333333335
Logistic Regression Accuracy: 0.19666666666666666


In [21]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.1, 1, 10],
    'class_weight': ['balanced', None]
}

grid = GridSearchCV(LinearSVC(), param_grid, cv=3)
grid.fit(X_train_tfidf, y_train)
print("Best Params:", grid.best_params_)
print("Best Score:", grid.best_score_)


Best Params: {'C': 0.1, 'class_weight': None}
Best Score: 0.20333333333333334


In [22]:
from sklearn.metrics import classification_report, accuracy_score


print("Naive Bayes Report:")
print(classification_report(y_test, nb.predict(X_test_tfidf)))

print("SVM Report:")
print(classification_report(y_test, svm.predict(X_test_tfidf)))


Naive Bayes Report:
               precision    recall  f1-score   support

      Bug Fix       0.15      0.13      0.14        62
Documentation       0.14      0.05      0.08        56
      Feature       0.20      0.36      0.26        64
      Testing       0.25      0.30      0.27        61
        UI/UX       0.15      0.11      0.12        57

     accuracy                           0.19       300
    macro avg       0.18      0.19      0.17       300
 weighted avg       0.18      0.19      0.18       300

SVM Report:
               precision    recall  f1-score   support

      Bug Fix       0.20      0.16      0.18        62
Documentation       0.14      0.16      0.15        56
      Feature       0.25      0.25      0.25        64
      Testing       0.31      0.30      0.30        61
        UI/UX       0.18      0.19      0.19        57

     accuracy                           0.21       300
    macro avg       0.21      0.21      0.21       300
 weighted avg       0.22    

In [23]:
import joblib


joblib.dump(svm, 'task_classifier_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

print("Task Classification Model and Vectorizer Saved!")


Task Classification Model and Vectorizer Saved!


In [24]:
clf = joblib.load('task_classifier_model.pkl')
vec = joblib.load('tfidf_vectorizer.pkl')

sample_text = ["Add login authentication feature"]
X_sample = vec.transform(sample_text)
pred = clf.predict(X_sample)
print("Predicted Category:", pred[0])


Predicted Category: UI/UX


In [25]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split


cat_encoder = LabelEncoder()
prio_encoder = LabelEncoder()

df['category_encoded'] = cat_encoder.fit_transform(df['category'])
df['priority_encoded'] = prio_encoder.fit_transform(df['priority'])


df['text'] = df['task_title'] + ' ' + df['task_description']


X_text = vectorizer.fit_transform(df['text'])


X_num = df[['category_encoded', 'estimated_hours']].values


from scipy.sparse import hstack
X_full = hstack((X_text, X_num))

y = df['priority_encoded']

X_train, X_test, y_train, y_test = train_test_split(X_full, y, test_size=0.2, random_state=42, stratify=y)


In [26]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print("RandomForest Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


RandomForest Accuracy: 0.25666666666666665
              precision    recall  f1-score   support

           0       0.24      0.22      0.23        76
           1       0.27      0.22      0.24        69
           2       0.28      0.40      0.33        84
           3       0.20      0.15      0.18        71

    accuracy                           0.26       300
   macro avg       0.25      0.25      0.25       300
weighted avg       0.25      0.26      0.25       300



In [27]:
joblib.dump(rf, 'priority_prediction_model.pkl')
joblib.dump(prio_encoder, 'priority_label_encoder.pkl')
print("Priority Prediction Model Saved Successfully!")


Priority Prediction Model Saved Successfully!


In [28]:

clf = joblib.load('task_classifier_model.pkl')
priority_model = joblib.load('priority_prediction_model.pkl')
vec = joblib.load('tfidf_vectorizer.pkl')


new_task = "Fix payment gateway API timeout issue"


task_vector = vec.transform([new_task])
pred_category = clf.predict(task_vector)[0]


cat_val = cat_encoder.transform([pred_category])[0]
estimated_hours = 5  # example
combined_input = hstack((task_vector, np.array([[cat_val, estimated_hours]])))

pred_priority = priority_model.predict(combined_input)[0]
final_priority = prio_encoder.inverse_transform([pred_priority])[0]

print(f"Predicted Category: {pred_category}")
print(f"Predicted Priority: {final_priority}")


Predicted Category: Testing
Predicted Priority: Low
